# 02 Modeling\n\nНоутбук фиксирует результаты экспериментов для отчета: baseline, улучшенные модели, метрики и выбор финальной модели. Обучение выполняется скриптом `src/train.py`, а здесь результаты загружаются из `artifacts/leaderboard.json`.

In [ ]:
from pathlib import Path\nimport json\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nPROJECT_ROOT = Path('..').resolve()\nleaderboard_path = PROJECT_ROOT / 'artifacts' / 'leaderboard.json'\nmetadata_path = PROJECT_ROOT / 'models' / 'model_metadata.json'\n\nwith leaderboard_path.open(encoding='utf-8') as file:\n    leaderboard = json.load(file)\nwith metadata_path.open(encoding='utf-8') as file:\n    metadata = json.load(file)

## Таблица сравнения моделей

In [ ]:
model_descriptions = {\n    'logistic_regression': 'Baseline: линейная модель с class_weight=balanced',\n    'random_forest': 'Improved: ансамбль деревьев с ограничением глубины',\n    'gradient_boosting': 'Improved: boosting деревьев решений',\n}\n\nmetrics = (\n    pd.DataFrame(leaderboard)\n    .T\n    .rename_axis('model')\n    .reset_index()\n)\nmetrics['description'] = metrics['model'].map(model_descriptions)\nmetrics['comment'] = metrics['model'].apply(\n    lambda name: 'финальная модель' if name == metadata['best_model_name'] else 'используется для сравнения'\n)\nmetrics[['model', 'description', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'comment']]

## Визуальное сравнение

In [ ]:
plot_df = metrics.set_index('model')[['precision', 'recall', 'f1', 'roc_auc']]\nax = plot_df.plot(kind='bar', figsize=(10, 5), rot=20)\nax.set_ylim(0, 1)\nax.set_title('Model comparison')\nax.set_ylabel('Metric value')\nax.grid(axis='y', alpha=0.25)\nplt.tight_layout()

## Вывод\n\nФинальная модель — `random_forest`. Она выбрана по основной метрике `F1` и дополнительно имеет высокий `Recall`, что важно для раннего выявления Jira issues с риском долгого закрытия.